# Dimensões de Qualidade

-----

``` Completude ```:
Percentual de campos críticos preenchidos (ex: **email**, **cpf**, **data_cadastro**).

``` Validade ```:
Regras de formato e domínio (regex para e-mail, datas válidas, faixas de idade).

``` Unicidade ```:
Deteccão de duplicidades por chave natural e chave composta.

``` Consistência ```:
Coerência entre colunas (ex.: **data_fim >= data_inicio**).

``` Atualidade (timeliness) ```:
Atraso entre evento de origem e ingestão.

``` Integridade referencial ```:
Chaves estrangeiras válidas entre tabelas.

In [0]:
# Imports
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
# Completude

df = spark.read.parquet('/Volumes/costumers_registrations/raw/registrations')

total_rows = df.count()

metrics = []

for c in df.columns:

  metrics.append(
    count(when(col(c).isNull(), True)).alias(c)
  )

result = df.agg(*metrics)

row = result.collect()[0]

print("Percentual de Campos Preenchidos\n")

for c in df.columns:
  nulls = row[c]
  percent = (1 - (nulls / total_rows)) * 100 if total_rows > 0 else 0
  print(f"{c}: {percent:.2f}%")

print("\n=====================================================\n")



In [0]:
# Validade

regex_email = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

result = df.select(
    (
        count(when(col("email").rlike(regex_email), True)) / count("*")
    ).alias("percent_regex_email_valido")
)

display(result)

In [0]:
display(df)